# Example 1: Basic LDA with Short Documents
## CISB5123 Text Analytics - Lab 9: Topic Modeling

This notebook demonstrates basic topic modeling using Latent Dirichlet Allocation (LDA) with short documents about tennis and politics.

## Cell 1: Import the libraries

In [1]:
# For text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# For topic modeling
from gensim import corpora
from gensim.models import LdaModel

# For data handling
import pandas as pd

# Download NLTK Resources
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

print("All libraries imported successfully!")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\mohda\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\mohda\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\mohda\AppData\Roaming\nltk_data...


All libraries imported successfully!


[nltk_data]   Package wordnet is already up-to-date!


## Cell 2: Load the Data

In [2]:
documents = [
    "Rafael Nadal Joins Roger Federer in Missing U.S. Open",
    "Rafael Nadal Is Out of the Australian Open",
    "Biden Announces Virus Measures",
    "Biden's Virus Plans Meet Reality",
    "Where Biden's Virus Plan Stands"
]

print("Documents loaded:")
for i, doc in enumerate(documents):
    print(f"{i}: {doc}")

Documents loaded:
0: Rafael Nadal Joins Roger Federer in Missing U.S. Open
1: Rafael Nadal Is Out of the Australian Open
2: Biden Announces Virus Measures
3: Biden's Virus Plans Meet Reality
4: Where Biden's Virus Plan Stands


## Cell 3: Preprocess the Data

This step:
- Tokenizes text into words
- Converts to lowercase
- Removes stopwords (common words like 'the', 'is')
- Lemmatizes words (converts to base form)

In [3]:
stop_words = set(stopwords.words('english'))  # Create a set of English stopwords
lemmatizer = WordNetLemmatizer()  # Initialize a WordNet lemmatizer

def preprocess_text(text):
    """
    Preprocess text by:
    1. Tokenizing into words and converting to lowercase
    2. Filtering out non-alphanumeric tokens
    3. Removing stopwords from the tokens
    4. Lemmatizing each token
    """
    tokens = word_tokenize(text.lower())  # Tokenize the text into words and convert to lowercase
    tokens = [token for token in tokens if token.isalnum()]  # Filter out non-alphanumeric tokens
    tokens = [token for token in tokens if token not in stop_words]  # Remove stopwords from the tokens
    tokens = [lemmatizer.lemmatize(token) for token in tokens]  # Lemmatize each token
    return tokens  # Return the preprocessed tokens

preprocessed_documents = [preprocess_text(doc) for doc in documents]  # Preprocess each document in the list

print("\nPreprocessed Documents:")
for i, doc in enumerate(preprocessed_documents):
    print(f"{i}: {doc}")


Preprocessed Documents:
0: ['rafael', 'nadal', 'join', 'roger', 'federer', 'missing', 'open']
1: ['rafael', 'nadal', 'australian', 'open']
2: ['biden', 'announces', 'virus', 'measure']
3: ['biden', 'virus', 'plan', 'meet', 'reality']
4: ['biden', 'virus', 'plan', 'stand']


## Cell 4: Create a document-term matrix

- **Dictionary**: Maps each unique word to a unique ID
- **Corpus (Bag of Words)**: Represents each document as word-frequency pairs

In [4]:
# Create a Gensim Dictionary object from the preprocessed documents
dictionary = corpora.Dictionary(preprocessed_documents)

print("\nDictionary created:")
print(f"Number of unique words: {len(dictionary)}")
print(f"Dictionary: {dictionary}")

# Convert each preprocessed document into a bag-of-words representation using the dictionary
corpus = [dictionary.doc2bow(doc) for doc in preprocessed_documents]

print("\nBag of Words (Corpus):")
for i, doc in enumerate(corpus):
    print(f"Document {i}: {doc}")


Dictionary created:
Number of unique words: 16
Dictionary: Dictionary<16 unique tokens: ['federer', 'join', 'missing', 'nadal', 'open']...>

Bag of Words (Corpus):
Document 0: [(0, 1), (1, 1), (2, 1), (3, 1), (4, 1), (5, 1), (6, 1)]
Document 1: [(3, 1), (4, 1), (5, 1), (7, 1)]
Document 2: [(8, 1), (9, 1), (10, 1), (11, 1)]
Document 3: [(9, 1), (11, 1), (12, 1), (13, 1), (14, 1)]
Document 4: [(9, 1), (11, 1), (13, 1), (15, 1)]


## Cell 5: Run LDA Model

**Parameters:**
- `num_topics=2`: Extract 2 topics
- `passes=15`: Iterate through the corpus 15 times for better results

In [5]:
# corpus: bag-of-words representation of the documents
# num_topics: number of topics to be extracted by the model
# id2word: dictionary mapping from word IDs to words
# passes: number of passes through the corpus during training

lda_model = LdaModel(corpus, num_topics=2, id2word=dictionary, passes=15)

print("\nLDA Model trained with 2 topics and 15 passes")


LDA Model trained with 2 topics and 15 passes


## Cell 6: Interpret Results - Get Dominant Topic for Each Document

In [6]:
# empty list to store dominant topic labels for each document
article_labels = []

# iterate over each processed document
for i, doc in enumerate(preprocessed_documents):
    # for each document, convert to box representation
    bow = dictionary.doc2bow(doc)
    # get list of topic probabilities
    topics = lda_model.get_document_topics(bow)
    # determine topic with highest probability
    dominant_topic = max(topics, key=lambda x: x[1])[0]
    # append to the list
    article_labels.append(dominant_topic)

# Create DataFrame
df = pd.DataFrame({"Article": documents, "Topic": article_labels})

# Print the DataFrame
print("Table with Articles and Topic:")
print(df)
print()

Table with Articles and Topic:
                                             Article  Topic
0  Rafael Nadal Joins Roger Federer in Missing U....      1
1         Rafael Nadal Is Out of the Australian Open      1
2                     Biden Announces Virus Measures      0
3                   Biden's Virus Plans Meet Reality      0
4                    Where Biden's Virus Plan Stands      0



## Cell 7: Display Top Terms for Each Topic

In [7]:
print("Top Terms for Each Topic:")
print("=" * 80)
for idx, topic in lda_model.print_topics():
    print(f"Topic {idx}:")
    terms = [term.strip() for term in topic.split("+")]
    for term in terms:
        weight, word = term.split("*")
        print(f"  - {word.strip()} (weight: {weight.strip()})")
    print()
    
print("\n**Topic 0:** Related to politics and virus (Biden, virus, plan)")
print("**Topic 1:** Related to tennis (Nadal, Rafael, Open, Federer)")

Top Terms for Each Topic:
Topic 0:
  - "virus" (weight: 0.166)
  - "biden" (weight: 0.166)
  - "plan" (weight: 0.119)
  - "meet" (weight: 0.071)
  - "reality" (weight: 0.071)
  - "stand" (weight: 0.071)
  - "announces" (weight: 0.071)
  - "measure" (weight: 0.071)
  - "australian" (weight: 0.024)
  - "open" (weight: 0.024)

Topic 1:
  - "rafael" (weight: 0.131)
  - "nadal" (weight: 0.131)
  - "open" (weight: 0.131)
  - "join" (weight: 0.079)
  - "missing" (weight: 0.079)
  - "federer" (weight: 0.079)
  - "roger" (weight: 0.079)
  - "australian" (weight: 0.079)
  - "biden" (weight: 0.027)
  - "virus" (weight: 0.027)


**Topic 0:** Related to politics and virus (Biden, virus, plan)
**Topic 1:** Related to tennis (Nadal, Rafael, Open, Federer)
